In [1]:
import pandas as pd

path_data = r"C:\Users\Highlightning\Documents\Ciencia de Datos\Proyectos\Clustering (Customer Segmentation & Persona Profiling)\data\raw\Online Retail.xlsx"

df_raw = pd.read_excel(path_data)

resumen = []

for col in df_raw.columns:
    resumen.append({
        "Col":      col,
        "Rows":     df_raw[col].count(),
        "dtype":    df_raw[col].dtypes,
        "NaN":      df_raw[col].isnull().sum(),
        "'0'":      df_raw[col].eq(0).sum(),
        "Unique":   df_raw[col].unique()[0:6].tolist(),
        "Unique_N": df_raw[col].nunique()
    })
df_resumen = pd.DataFrame(resumen)
pd.set_option("display.max_colwidth", None)
df_resumen

,Col,Rows,dtype,NaN,'0',Unique,Unique_N
0,InvoiceNo,541909,object,0,0,"[536365, 536366, 536367, 536368, 536369, 536370]",25900
1,StockCode,541909,object,0,0,"[85123A, 71053, 84406B, 84029G, 84029E, 22752]",4070
2,Description,540455,object,1454,0,"[WHITE HANGING HEART T-LIGHT HOLDER, WHITE METAL LANTERN, CREAM CUPID HEARTS COAT HANGER, KNITTED UNION FLAG HOT WATER BOTTLE, RED WOOLLY HOTTIE WHITE HEART., SET 7 BABUSHKA NESTING BOXES]",4223
3,Quantity,541909,int64,0,0,"[6, 8, 2, 32, 3, 4]",722
4,InvoiceDate,541909,datetime64[ns],0,0,"[2010-12-01 08:26:00, 2010-12-01 08:28:00, 2010-12-01 08:34:00, 2010-12-01 08:35:00, 2010-12-01 08:45:00, 2010-12-01 09:00:00]",23260
5,UnitPrice,541909,float64,0,2515,"[2.55, 3.39, 2.75, 7.65, 4.25, 1.85]",1630
6,CustomerID,406829,float64,135080,0,"[17850.0, 13047.0, 12583.0, 13748.0, 15100.0, 15291.0]",4372
7,Country,541909,object,0,0,"[United Kingdom, France, Australia, Netherlands, Germany, Norway]",38


In [2]:
""" New Features """
import numpy as np
df_raw["CustomerID"] = df_raw["CustomerID"].replace(np.nan, 99999.0)
df_raw["Return"] = np.where(df_raw["Quantity"]<0, 1, 0)
df_raw["Customer_Type"] = np.where(df_raw["CustomerID"]==99999.0, "Customer_Final", "Customer_ID")
df_raw["Description_Error"] = np.where(df_raw["Description"].isnull(), 1, 0)
df_raw["Gift"] = np.where(df_raw["UnitPrice"].eq(0), 1, 0)
df_raw["Total_Invoice"] = df_raw["Quantity"] * df_raw["UnitPrice"]

In [3]:
df_raw[(df_raw["UnitPrice"]==0)&(df_raw["Description"].isnull())].shape

(1454, 13)

In [4]:
df_raw[df_raw["UnitPrice"]<0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Return,Customer_Type,Description_Error,Gift,Total_Invoice
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,99999.0,United Kingdom,0,Customer_Final,0,0,-11062.06
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,99999.0,United Kingdom,0,Customer_Final,0,0,-11062.06


In [5]:
""" New dataset """

df_clean = df_raw.copy()
df_clean = df_clean.drop(df_clean[df_clean["Customer_Type"].eq("Customer_Final")].index)
df_clean = df_clean.drop(df_clean[df_clean["Return"].eq(1)].index)
df_clean = df_clean.drop(df_clean[df_clean["Gift"].eq(1)].index)
df_clean = df_clean.drop(columns=["Return", "Customer_Type", "Description_Error", "Gift"])

In [6]:
""" 🔍 Tratamiento de Outliers (Valores Atípicos Extreme) """

df_clean[["Quantity", "UnitPrice", "Total_Invoice"]].describe(include="all")

,Quantity,UnitPrice,Total_Invoice
count,397884.000000,397884.000000,397884.000000
mean,12.988238,3.116488,22.397000
std,179.331775,22.097877,309.071041
min,1.000000,0.001000,0.001000
25%,2.000000,1.250000,4.680000
50%,6.000000,1.950000,11.800000
75%,12.000000,3.750000,19.800000
max,80995.000000,8142.750000,168469.600000


In [ ]:
q_limit = df_clean["Quantity"].quantile(0.999)
m_limit = df_clean["Total_Invoice"].quantile(0.999)

print(f"\nLímite percentil 99.9 para Cantidad: {q_limit}")
print(f"Límite percentil 99.9 para Monto: {m_limit}")


Límite percentil 99.9 para Cantidad: 504.0
Límite percentil 99.9 para Monto: 876.0


In [8]:
""" Dataset final """
df_final = df_clean[(df_clean["Quantity"]<=q_limit) & (df_clean["Total_Invoice"]<=m_limit)]

print(f"Registros de df_raw:    {len(df_raw):,}")
print(f"Registros de df_clean:  {len(df_clean):,}")
print(f"Registros de df_final:  {len(df_final):,}")
print(f"Total de registros depurados: {len(df_raw)-len(df_final):,}")

Registros de df_raw:    541,909
Registros de df_clean:  397,884
Registros de df_final:  397,236
Total de registros depurados: 144,673


In [11]:
""" Exportar archivo trabajado """
df_final.to_csv(rf"C:\Users\Highlightning\Documents\Ciencia de Datos\Proyectos\Clustering (Customer Segmentation & Persona Profiling)\data\processed/clean_retail.csv", index=False)
print("¡Archivo 'clean_retail.csv' guardado en data/processed/!")

¡Archivo 'clean_retail.csv' guardado en data/processed/!
